In [1]:
import os, json, time, datetime as dt, csv, pathlib
from typing import Dict, List
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

DATA_RAW = pathlib.Path("data/raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)

load_dotenv()
ALPHA_KEY = os.getenv("ALPHAVANTAGE_API_KEY")
print("Loaded ALPHAVANTAGE_API_KEY?", bool(ALPHA_KEY))
#print isi API nya
print(ALPHA_KEY)

Loaded ALPHAVANTAGE_API_KEY? True
DPO5UNECL0FW8CXS


In [2]:
def safe_stamp():
    return dt.datetime.now().strftime("%Y%m%d-%H%M%S")

def safe_filename(prefix: str, meta: Dict[str, str]) -> str:
    mid = "_".join([f"{k}-{str(v).replace(' ', '-')[:20]}" for k, v in meta.items()])
    return f"{prefix}_{mid}_{safe_stamp()}.csv"

def validate_df(df: pd.DataFrame, required_cols: List[str], dtypes_map: Dict[str, str]) -> Dict[str, str]:
    msgs = {}
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        msgs['missing_cols'] = f"Missing columns: {missing}"
    for col, dtype in dtypes_map.items():
        if col in df.columns:
            try:
                if dtype == 'datetime64[ns]':
                    pd.to_datetime(df[col])
                elif dtype == 'float':
                    pd.to_numeric(df[col])
            except Exception as e:
                msgs[f'dtype_{col}'] = f"Failed to coerce {col} to {dtype}: {e}"
    na_counts = df.isna().sum().sum()
    msgs['na_total'] = f"Total NA values: {na_counts}"
    return msgs

In [15]:
SYMBOL = "AAPL"
use_alpha = bool(ALPHA_KEY)
print("Using Alpha Vantage:", use_alpha)

if use_alpha:
    url = "https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=AAPL&apikey=QNF78IHU7X1LZ9KJ"
    r = requests.get(url)
    js = r.json()
    key = [k for k in js.keys() if "Time Series" in k]
    assert key, f"Unexpected response keys: {list(js.keys())}"
    series = js[key[0]]
    df_api = (pd.DataFrame(series).T
              .rename_axis('date')
              .reset_index())
    # keep a couple columns and coerce types
    df_api = df_api[['date', '4. close']].rename(columns={'4. close': 'close'})
    df_api['date'] = pd.to_datetime(df_api['date'])
    df_api['close'] = pd.to_numeric(df_api['close'])
else:
    import yfinance as yf
    df_api = yf.download(SYMBOL, period="6mo", interval="1d").reset_index()[['Date','Adj Close']]
    df_api.columns = ['date','close']

df_api = df_api.sort_values('date').reset_index(drop=True)
msgs = validate_df(df_api, required_cols=['date','close'], dtypes_map={'date':'datetime64[ns]','adj_close':'float'})
print(msgs)

fname = safe_filename(prefix="api", meta={"source": "alpha" if use_alpha else "yfinance", "symbol": SYMBOL})
out_path = DATA_RAW / fname
df_api.to_csv(out_path, index=False)
print("Saved:", out_path)

Using Alpha Vantage: True
{'na_total': 'Total NA values: 0'}
Saved: data\raw\api_source-alpha_symbol-AAPL_20250819-131732.csv


In [ ]:
SCRAPE_URL = "https://example.com/markets-table"  # replace with permitted page
headers = {"User-Agent": "AFE-Course-Notebook/1.0 (contact: instructor@example.edu)"}
try:
    resp = requests.get(SCRAPE_URL, headers=headers, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    table = soup.find('table')
    rows = []
    for tr in table.find_all('tr'):
        cells = [td.get_text(strip=True) for td in tr.find_all(['td','th'])]
        if cells:
            rows.append(cells)
    # assume first row is header
    header, *data = rows
    df_scrape = pd.DataFrame(data, columns=header)
except Exception as e:
    print("Scrape failed (demoing with inline HTML).", e)
    html = """
    <table>
      <tr><th>Ticker</th><th>Price</th></tr>
      <tr><td>AAA</td><td>101.2</td></tr>
      <tr><td>BBB</td><td>98.7</td></tr>
    </table>
    """
    soup = BeautifulSoup(html, 'html.parser')
    rows = []
    for tr in soup.find_all('tr'):
        cells = [td.get_text(strip=True) for td in tr.find_all(['td','th'])]
        if cells:
            rows.append(cells)
    header, *data = rows
    df_scrape = pd.DataFrame(data, columns=header)

if 'Price' in df_scrape.columns:
    df_scrape['Price'] = pd.to_numeric(df_scrape['Price'], errors='coerce')

msgs2 = validate_df(df_scrape, required_cols=list(df_scrape.columns), dtypes_map={})
print(msgs2)

fname2 = safe_filename(prefix="scrape", meta={"site": "example", "table": "markets"})
out_path2 = DATA_RAW / fname2
df_scrape.to_csv(out_path2, index=False)
print("Saved:", out_path2)

In [13]:
# 股票符号
SYMBOL = "AAPL"

# 使用 Alpha Vantage API
print("Using Alpha Vantage")
url = "https://www.alphavantage.co/query"
params = {
    "function": "TIME_SERIES_DAILY",
    "symbol": SYMBOL,
    "outputsize": "full",
    "apikey": ALPHA_KEY,
    "datatype": "json"
}

# 发送请求
r = requests.get(url, params=params, timeout=30)
r.raise_for_status()
data = r.json()

# 提取时间序列数据
time_series = data.get("Time Series (Daily)", {})
if not time_series:
    raise ValueError("No 'Time Series (Daily)' found in API response")

# 转换数据为DataFrame
rows = []
for date, values in time_series.items():
    row = {
        "date": date,
        "open": float(values["1. open"]),
        "high": float(values["2. high"]),
        "low": float(values["3. low"]),
        "close": float(values["4. close"]),
        "volume": int(values["5. volume"])
    }
    rows.append(row)

df_api = pd.DataFrame(rows)

# 数据清洗
df_api["date"] = pd.to_datetime(df_api["date"])
df_api = df_api.sort_values("date", ascending=False).reset_index(drop=True)

# 数据验证
msgs = validate_df(
    df_api, 
    required_cols=["date", "open", "high", "low", "close", "volume"],
    dtypes_map={
        "date": "datetime64[ns]",
        "open": "float",
        "high": "float",
        "low": "float",
        "close": "float",
        "volume": "float"
    }
)
print("Validation messages:", msgs)

# 保存数据
fname = safe_filename(
    prefix="api_daily", 
    meta={"source": "alphavantage", "symbol": SYMBOL}
)
out_path = DATA_RAW / fname
df_api.to_csv(out_path, index=False)
print(f"Saved {len(df_api)} records to: {out_path}")

# 打印前5行数据
print("\nSample data:")
print(df_api.head())

Using Alpha Vantage
Validation messages: {'na_total': 'Total NA values: 0'}
Saved 6488 records to: data\raw\api_daily_source-alphavantage_symbol-AAPL_20250819-125210.csv

Sample data:
        date     open    high      low   close    volume
0 2025-08-18  231.700  233.12  230.110  230.89  37476188
1 2025-08-15  234.000  234.28  229.335  231.59  56038657
2 2025-08-14  234.055  235.12  230.850  232.78  51916275
3 2025-08-13  231.070  235.00  230.430  233.33  69878546
4 2025-08-12  228.005  230.80  227.070  229.65  55672301


In [16]:
# 修改为 Yahoo Finance AAPL 页面
SCRAPE_URL = "https://finance.yahoo.com/quote/AAPL/history?p=AAPL"  # AAPL 历史数据页面
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"}

try:
    print(f"Scraping AAPL data from {SCRAPE_URL}")
    resp = requests.get(SCRAPE_URL, headers=headers, timeout=30)
    resp.raise_for_status()
    
    # 解析页面内容
    soup = BeautifulSoup(resp.text, 'html.parser')
    
    # 定位历史数据表格
    table = soup.find('table', {'data-test': 'historical-prices'})
    if not table:
        raise ValueError("Historical prices table not found")
    
    # 提取表头
    header_row = table.find('thead').find('tr')
    headers = [th.get_text(strip=True) for th in header_row.find_all('th')]
    
    # 提取数据行
    rows = []
    for tr in table.find('tbody').find_all('tr'):
        # 跳过包含股息信息的行
        if 'Dividend' in tr.get_text():
            continue
            
        cells = [td.get_text(strip=True) for td in tr.find_all('td')]
        if cells and len(cells) == len(headers):
            rows.append(cells)
    
    # 创建DataFrame
    df_scrape = pd.DataFrame(rows, columns=headers)
    
    # 重命名列以匹配标准格式
    column_map = {
        'Date': 'date',
        'Open': 'open',
        'High': 'high',
        'Low': 'low',
        'Close*': 'close',
        'Adj Close**': 'adj_close',
        'Volume': 'volume'
    }
    df_scrape = df_scrape.rename(columns=column_map)
    
    # 过滤掉包含Dividend的行（如果有漏网之鱼）
    df_scrape = df_scrape[df_scrape['open'] != 'Dividend']
    
    # 转换数据类型
    for col in ['open', 'high', 'low', 'close', 'adj_close']:
        df_scrape[col] = df_scrape[col].str.replace(',', '').astype(float)
    
    df_scrape['volume'] = df_scrape['volume'].str.replace(',', '').astype(int)
    df_scrape['date'] = pd.to_datetime(df_scrape['date'])
    
    print(f"Successfully scraped {len(df_scrape)} rows of AAPL data")

except Exception as e:
    print(f"Scrape failed: {e}. Using fallback data.")
    # 使用备用数据（示例数据）
    fallback_data = {
        'date': ['2025-08-18', '2025-08-17', '2025-08-16'],
        'open': [230.50, 229.80, 228.25],
        'high': [232.75, 231.20, 229.90],
        'low': [229.30, 228.50, 227.10],
        'close': [231.20, 230.10, 228.75],
        'adj_close': [231.20, 230.10, 228.75],
        'volume': [45678900, 42345600, 39876500]
    }
    df_scrape = pd.DataFrame(fallback_data)
    df_scrape['date'] = pd.to_datetime(df_scrape['date'])

# 数据验证
msgs2 = validate_df(df_scrape, 
                   required_cols=['date', 'open', 'high', 'low', 'close', 'adj_close', 'volume'],
                   dtypes_map={
                       'date': 'datetime64[ns]',
                       'open': 'float',
                       'high': 'float',
                       'low': 'float',
                       'close': 'float',
                       'adj_close': 'float',
                       'volume': 'int'
                   })
print("Validation messages:", msgs2)

# 保存数据
fname2 = safe_filename(prefix="scrape", meta={"site": "yahoo", "symbol": "AAPL"})
out_path2 = DATA_RAW / fname2
df_scrape.to_csv(out_path2, index=False)
print(f"Saved scraped data to: {out_path2}")

# 打印前5行数据
print("\nScraped AAPL data sample:")
print(df_scrape.head())

Scraping AAPL data from https://finance.yahoo.com/quote/AAPL/history?p=AAPL
Scrape failed: 404 Client Error: Not Found for url: https://finance.yahoo.com/quote/AAPL/history/?p=AAPL. Using fallback data.
Validation messages: {'na_total': 'Total NA values: 0'}
Saved scraped data to: data\raw\scrape_site-yahoo_symbol-AAPL_20250819-133736.csv

Scraped AAPL data sample:
        date    open    high    low   close  adj_close    volume
0 2025-08-18  230.50  232.75  229.3  231.20     231.20  45678900
1 2025-08-17  229.80  231.20  228.5  230.10     230.10  42345600
2 2025-08-16  228.25  229.90  227.1  228.75     228.75  39876500
